---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: Web Analytics

### 📋 **Topic**: Using LLMs

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.



---

In this Lecture, we'll go over many tools that will help us use LLMs more easily and effectively in our projects. In particular, we will cover:

1. **Speeding things up** using Async Programming to speed things up
2. **Making our code robust** to errors and rate limits with retries
3. **Structured outputs** using Pydantic
4. **Unifying LLM API calls** using LiteLLM
5. **Processing images** with VLMs


---

## 🔀 1. Using Async Programming to speed things up

Instead of sending prompts one by one, you can send multiple prompts asynchronously.
- We'll see what this means by example.

Let's say you have 10 prompts you want to process.

In [ ]:
import openai
import os
import time # we'll use this to time our operations
from dotenv import load_dotenv 

load_dotenv() # expose our API key from our .env as an environment variable
openai.api_key = os.getenv("OPENAI_API_KEY")

prompts = [
    "What is the capital of France?",
    "What is the capital of Greece?",
    "What is the capital of Bulgaria?",
    "What is the capital of Panama?",
    "What is the capital of Pakistan?",
    "What is the capital of Egypt?",
    "What is the capital of India?",
    "What is the capital of China?",
    "What is the capital of Brazil?",
    "What is the capital of Argentina?",
    "What is the capital of Mexico?"
]



What we did last week, was to wait for each response to come back before sending the next one.

This is called **sequential processing**.

In [ ]:
# Inefficient: sending one by one
start_time = time.time()
results_one_by_one = []
for prompt in prompts:
    response = openai.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}]
    )
    results_one_by_one.append(response.choices[0].message.content)
    print(f"Prompt: {prompt}")
    print(f"Response: {response.choices[0].message.content}\n")
end_time = time.time()
print(f"Sequential processing took: {end_time - start_time:.2f} seconds")


When we write code normally (called **synchronous/sequential code**), Python does one thing at a time:
- Send a request
- Wait for response… 
- Response arrives
- Move to the next request

This is simple, but slow — especially when each request takes 1–2 seconds. Examples of slow requests are:
- Reading from a database
- Reading from a file
- **Making an API call**
  
**Async** (short for asynchronous) means that we can start many independent tasks at the same time.
- We will wait for all of them to finish, but we can start them all at once.
- This is perfect for doing things such as calling the OpenAI API, because the model is doing most of the work, while your computer is just waiting


When a function is marked with:

```python
async def ...
```

it means that
- The function can be paused while it waits (for example, waiting for the OpenAI API to respond)
- Python can move on to other tasks while the function is paused
- Later, when the response comes back, Python continues the function

Inside async functions, we use:
```python
await ...
```
to pause the function until the awaited task is finished.





In [ ]:
import asyncio # This is a library that allows us to run code asynchronously
from openai import AsyncOpenAI # This is the async version of the OpenAI API

client = AsyncOpenAI()

# note that this function definition begins with "async"
# this means that the function can be run asynchronously
async def get_completion(prompt: str):
    resp = await client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.choices[0].message.content


async def get_completions(prompts):
    start = time.time()

    # Fire all requests concurrently
    tasks = [asyncio.create_task(get_completion(prompt)) for prompt in prompts]
    results = await asyncio.gather(*tasks)

    # Print results
    for prompt, answer in zip(prompts, results):
        print(f"Prompt: {prompt}")
        print(f"Response: {answer}\n")

    end = time.time()
    print(f"Concurrent processing took: {end - start:.2f} seconds")

    return results


In [ ]:
await get_completions(prompts)


---

## 🔁 2. Making Our LLM Calls Robust

When we send many requests to the OpenAI API, most of them succeed immediately.
But in real projects — even simple ones — things occasionally go wrong.

There are two main categories of issues:

**1. Temporary Errors** 

These include things like:
- internet hiccups (Wi-Fi drops, DNS failure)
- server issues (OpenAI responds with a 500-level error)
- connection timeouts
These errors have nothing to do with your code. 

**2. Rate Limits (HTTP 429)**

OpenAI (and every API) prevents users from sending too many requests too fast and consuming too many resources.
- If you exceed the limit, you’ll get a `429: Too Many Requests` error


**Solving this with retries  and exponential backoff**

Retries help us handle both error categories:
- If the API says “slow down” → we back off and retry later
- If a temporary network issue occurs → retry fixes it
- If the API is busy → retry after waiting

The strategy we use is called **exponential backoff + jitter**
- after each failure, we wait a little longer than the last time
- we add a small random delay (“jitter”) to avoid many tasks retrying at the same moment

For example, if we have a max of 5 retries, with a base wait time of 1 second, we would wait:
```text
1st retry → wait ~1 second + a tiny random delay  
2nd retry → wait ~2 seconds + a tiny random delay  
3rd retry → wait ~4 seconds + a tiny random delay  
4th retry → wait ~8 seconds + a tiny random delay  
5th retry → wait ~16 seconds + a tiny random delay  
fail 
```

This strategy is very common in real production systems.









To see the retry pattern clearly, 
- we'll first create a **fake API function** that always fails with high probability, and throws a custom `TemporaryAPIError`.  
- we'll then implement a retry loop with exponential backoff and try to call the fake API.

In [ ]:
import time
import random

class TemporaryAPIError(Exception):
    """Simulates a retryable API error."""
    pass


def fake_llm_call(prompt: str) -> str:
    """
    Fake function that succeeds with 10% probability and fails with 90% probability.
    """
    if random.random() < 0.10:   # 10% success probability
        print(f"fake_llm_call({prompt!r}) → SUCCESS!")
        return f"Fake answer to: {prompt}"
    else:
        print(f"fake_llm_call({prompt!r}) → FAIL")
        raise TemporaryAPIError("429 Too Many Requests (fake)")


def call_with_retries(prompt: str, max_retries: int = 5, base_delay: float = 1.0):
    """
    Retry wrapper with exponential backoff.
    Tries max_retries times in total.
    """
    for attempt in range(1, max_retries + 1):
        try:
            print(f"\nAttempt {attempt}...")
            return fake_llm_call(prompt)

        except TemporaryAPIError as e:
            print(f"  Temporary error: {e}")

            if attempt == max_retries:
                print("  ❌ Out of retries. Giving up.")
                raise

            # exponential backoff + jitter
            delay = base_delay * (2 ** (attempt - 1)) + random.random()
            print(f"  Waiting {delay:.1f} seconds before retrying...")
            time.sleep(delay)


# Run it a few times to see successes and failures
try:
    result = call_with_retries("hello world", max_retries=5)
    print("\nFinal result:", result)
except TemporaryAPIError:
    print("\nFinal result: failed after all retries.")


**Using `tenacity`** 

Instead of writing our own retry loop, we can let the `tenacity` library handle retries for us. 
- We simply "decorate" our function with a retry policy.
- A decorator is a function that takes another function and returns a new version of it with some added functionality.

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(
    retry=retry_if_exception_type(TemporaryAPIError),
    stop=stop_after_attempt(5),                 # 5 attempts total
    wait=wait_exponential(multiplier=1, min=1), # 1s, 2s, 4s, 8s, 16s backoff
    reraise=True,                               # re-raise the error if all attempts fail
)
def fake_llm_call_with_tenacity(prompt: str) -> str:
    """Tenacity handles retries for us."""
    return fake_llm_call(prompt)


# Run it a couple times to see the retries
try:
    result = fake_llm_call_with_tenacity("hello from tenacity")
    print("\nFinal result:", result)
except TemporaryAPIError:
    print("\nFinal result: failed after all retries.")

OpenAI's client has a retry policy built in, but it's not as flexible as tenacity's.

In [ ]:
from openai import OpenAI

client = OpenAI(max_retries=5, timeout=30)

response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "user", "content": "Hello, world!"}],
)

print(response.choices[0].message.content)


---

## 📋 3. Structured Outputs with Pydantic

When we call LLMs, we usually get back free-form text. But often, we want structured data — like JSON objects with specific fields and types.

**Why structured outputs?**
- **Reliability**: Guaranteed format makes parsing easier and more reliable
- **Type safety**: We know exactly what fields exist and their types
- **Validation**: Invalid responses are caught automatically
- **Integration**: Easy to use with databases, APIs, and other systems

**The problem with unstructured outputs:**

Let's say we want to extract information about a movie review. Without structure, we might get:
- "The review is positive. The rating is 4.5 stars."
- "Rating: 4.5/5, Sentiment: positive"
- "Positive review, 4.5 stars"

Each format is different, making it hard to parse consistently.

**The solution: Pydantic**

Pydantic is a Python library that lets us define data models with types and validation. OpenAI's API can return responses that match these models exactly.

### 3.1 Defining a Pydantic model

First, we define what structure we want using a Pydantic model:


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# Define the structure we want
class MovieReview(BaseModel):
    """Information extracted from a movie review"""
    sentiment: Literal["positive", "negative", "neutral"] = Field(
        description="The overall sentiment of the review"
    )
    rating: float = Field(
        description="Numeric rating from 1.0 to 5.0",
        ge=1.0,
        le=5.0
    )
    key_points: list[str] = Field(
        description="List of main points mentioned in the review",
        min_length=1,
        max_length=5
    )
    reviewer_name: str | None = Field(
        default=None,
        description="Name of the reviewer if mentioned"
    )


### 3.2 Using structured outputs with OpenAI

Now we can ask the LLM to return data in this exact format:


In [ ]:
from openai import OpenAI

client = OpenAI()

review_text = """
This movie was absolutely fantastic! The cinematography was stunning, 
and the acting performances were top-notch. I'd give it 4.5 stars. 
The plot kept me engaged from start to finish. Highly recommend!
- Sarah Johnson
"""

# Request structured output
response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "Extract information from movie reviews. Return structured data."
        },
        {
            "role": "user",
            "content": f"Extract information from this review:\n\n{review_text}"
        }
    ],
    response_format=MovieReview  # This tells OpenAI to return data matching our model
)

# The response is automatically parsed into our Pydantic model
review_data = response.choices[0].message.parsed

print(f"Sentiment: {review_data.sentiment}")
print(f"Rating: {review_data.rating}")
print(f"Key Points: {review_data.key_points}")
print(f"Reviewer: {review_data.reviewer_name}")
print(f"\nFull object: {review_data}")
print(f"\nAs JSON: {review_data.model_dump_json(indent=2)}")


### 3.3 Processing multiple reviews

We can easily process multiple reviews and get structured data for each:


In [ ]:
reviews = [
    "Terrible movie. Boring plot, bad acting. 1.5 stars. - John Doe",
    "It was okay. Nothing special, but not terrible either. 3 stars.",
    "Amazing cinematography and great performances! 4.8 stars. Highly recommend! - Jane Smith"
]

# Process all reviews
review_data_list = []

for review_text in reviews:
    response = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": "Extract information from movie reviews. Return structured data."
            },
            {
                "role": "user",
                "content": f"Extract information from this review:\n\n{review_text}"
            }
        ],
        response_format=MovieReview
    )
    
    review_data_list.append(response.choices[0].message.parsed)

# Now we have structured data we can work with
for i, data in enumerate(review_data_list, 1):
    print(f"Review {i}:")
    print(f"  Sentiment: {data.sentiment}")
    print(f"  Rating: {data.rating}")
    print(f"  Reviewer: {data.reviewer_name or 'Anonymous'}")
    print()


### 3.4 More complex examples

Pydantic models can be nested and include more complex structures:


In [ ]:
from typing import Optional

class Actor(BaseModel):
    """Information about an actor"""
    name: str
    role: str
    performance_rating: float = Field(ge=1.0, le=5.0)

class MovieInfo(BaseModel):
    """Comprehensive movie information"""
    title: str
    genre: list[str]
    release_year: Optional[int] = None
    director: Optional[str] = None
    main_actors: list[Actor] = Field(default_factory=list)
    plot_summary: str
    themes: list[str] = Field(default_factory=list)

# Example: Extract structured movie information
movie_description = """
The Matrix (1999) is a science fiction action film directed by the Wachowskis.
It stars Keanu Reeves as Neo, a computer programmer who discovers reality is a simulation.
Laurence Fishburne plays Morpheus, who guides Neo. Carrie-Anne Moss plays Trinity.
The film explores themes of reality, freedom, and choice. It's a cyberpunk thriller.
"""

response = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": "Extract structured information about movies from descriptions."
        },
        {
            "role": "user",
            "content": f"Extract information about this movie:\n\n{movie_description}"
        }
    ],
    response_format=MovieInfo
)

movie_data = response.choices[0].message.parsed

print(f"Title: {movie_data.title}")
print(f"Genre: {', '.join(movie_data.genre)}")
print(f"Release Year: {movie_data.release_year}")
print(f"Director: {movie_data.director}")
print("\nActors:")
for actor in movie_data.main_actors:
    print(f"  - {actor.name} as {actor.role} (rating: {actor.performance_rating})")
print(f"\nThemes: {', '.join(movie_data.themes)}")


### 3.6 Benefits and best practices

**Key benefits:**
- **Type safety**: Your IDE can autocomplete fields and catch errors
- **Validation**: Pydantic automatically validates the data matches your schema
- **Documentation**: The model serves as documentation of what data you expect
- **Consistency**: Every response follows the same structure

**Best practices:**
- Use `Field()` to add descriptions — these help the LLM understand what to extract
- Use `Literal` types for constrained choices (like sentiment categories)
- Make optional fields explicit with `Optional` or `| None`
- Use nested models for complex structures
- Add validation constraints (like `ge`, `le` for numeric ranges)

**When to use:**
- Extracting structured data from unstructured text
- Building APIs that need consistent response formats
- Data processing pipelines where you need reliable parsing
- Any time you need guaranteed data structure

---

## 🔌 4. Unifying LLM API calls with LiteLLM

So far, we've been using OpenAI's API directly. But what if you want to:
- Switch between different providers (OpenAI, Anthropic, Google, etc.) easily?
- Use open-source models locally?
- Have a consistent interface regardless of the provider?
- Manage costs by routing to cheaper models when appropriate?

**LiteLLM** is a library that provides a unified interface for calling many different LLM providers. You write code once, and it works with OpenAI, Anthropic, Google, Cohere, and many others.

**Why use LiteLLM?**
- **Provider agnostic**: Same code works with different providers
- **Easy switching**: Change providers by just changing the model name
- **Cost management**: Built-in support for tracking and managing costs
- **Fallbacks**: Automatically fall back to another provider if one fails
- **Open source models**: Works with local models and open-source alternatives

---


### 4.1 Basic usage

With LiteLLM, you use the same interface regardless of the provider:


In [ ]:
# pip install litellm

import litellm
import os
from dotenv import load_dotenv

load_dotenv()

# Set your API keys (assumes ANTHROPIC_API_KEY and GOOGLE_API_KEY are in .env)
# litellm.set_verbose = True  # Uncomment to see detailed logs

# The model name format is: provider/model-name
# Anthropic Claude
response_claude = litellm.completion(
    model="claude-sonnet-4-5-20250929",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)
print("Claude:", response_claude.choices[0].message.content)
print()

# Google Gemini
response_gemini = litellm.completion(
    model="gemini/gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ]
)
print("Gemini:", response_gemini.choices[0].message.content)


Another example: the same code works with different providers — just change the model name:


In [ ]:
# Same function, different providers
prompt = "Explain quantum computing in one sentence."

# Anthropic Claude
response_claude = litellm.completion(
    model="claude-sonnet-4-5-20250929", 
    messages=[{"role": "user", "content": prompt}]
)
print("Claude:", response_claude.choices[0].message.content)
print()

# Google Gemini
response_gemini = litellm.completion(
    model="gemini/gemini-3-pro-preview", 
    messages=[{"role": "user", "content": prompt}]
)
print("Gemini:", response_gemini.choices[0].message.content)
print()



### 4.3 Using async with LiteLLM

LiteLLM works seamlessly with async:


In [ ]:
import asyncio

async def get_response_async(prompt: str, model: str):
    """Get a response asynchronously using LiteLLM"""
    response = await litellm.acompletion(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

# Process multiple prompts concurrently using different providers
prompts = [
    "What is machine learning?",
    "What is deep learning?",
    "What is natural language processing?"
]

# Use different providers for each prompt
models = [
    "claude-sonnet-4-5-20250929", # Anthropic
    "gemini/gemini-3-pro-preview", # Google
    "openai/gpt-5.1" # OpenAI
]

async def process_all():
    tasks = [
        get_response_async(prompt, model) 
        for prompt, model in zip(prompts, models)
    ]
    results = await asyncio.gather(*tasks)
    for prompt, model, result in zip(prompts, models, results):
        print("-"*100)
        print(f"Model: {model}")
        print(f"-"*100)
        print(f"Q: {prompt}")
        print(f"A: {result}\n")

# Run it
await process_all()


### 4.4 Fallbacks and error handling

LiteLLM can automatically fall back to another provider if one fails:


In [ ]:
# Define a fallback chain: try Claude first, then Gemini, then Claude Haiku (cheaper)
response = litellm.completion(
    model="openai/gpt-5.1",  # Primary model (OpenAI)
    messages=[{"role": "user", "content": "Explain async programming in one sentence."}],
    fallbacks=["gemini/gemini-3-pro-preview", "claude-sonnet-4-5-20250929"]  # Fallback models
)

print(response.choices[0].message.content)
print(f"\nModel used: {response.model}")
print("If the primary model fails, it will automatically try the fallback models in order")


### 4.5 Cost tracking

LiteLLM provides built-in cost tracking and logging capabilities. This is crucial for:
- **Budget management**: Know how much you're spending
- **Cost optimization**: Compare costs across providers
- **Debugging**: Track what requests were made and their outcomes
- **Analytics**: Understand usage patterns

First, let's see how to track costs:

In [ ]:
response = litellm.completion(
    model="claude-sonnet-4-5-20250929",
    messages=[{"role": "user", "content": "What is Python?"}]
)

# Access cost information
print(f"Response: {response.choices[0].message.content}\n")
print("=== Cost Information ===")
print(f"Model: {response.model}")
print(f"Input tokens: {response.usage.prompt_tokens if hasattr(response, 'usage') else 'N/A'}")
print(f"Output tokens: {response.usage.completion_tokens if hasattr(response, 'usage') else 'N/A'}")
print(f"Total tokens: {response.usage.total_tokens if hasattr(response, 'usage') else 'N/A'}")

# LiteLLM calculates costs automatically
if hasattr(response, '_hidden_params') and 'response_cost' in response._hidden_params:
    print(f"Estimated cost: ${response._hidden_params.get('response_cost', 'N/A')}")


### 4.6 Logging

Please see the documentation for more details: https://docs.litellm.ai/docs/

---

## 5. VLMS

We've been working with LLMs that only process text, but many modern LLMs can also process images! These are called **Vision Language Models (VLMs)** or **multimodal models**.

With image understanding, you can accomplish a wide range of tasks:
- **image description**: Describe what's in an image
- **visual question answering**: Answer questions about images
- **image analysis**: Extract information, detect objects, read text
- **content moderation**: Check if images are appropriate
- **document understanding**: Read and understand documents, charts, diagrams

### 5.1 Basic image processing

First, let's see how to send an image to a VLM. We can use either:
- A URL to an image on the web
- A local file path
- Base64-encoded image data


### 5.2 Using local images

To use a local image file, we need to encode it as base64:


In [ ]:
import litellm
import base64
from dotenv import load_dotenv
load_dotenv()

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

image_path = "../assets/wally.png"
base64_image = encode_image(image_path)

# Universal multimodal message
vision_message = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Where is Wally?"},
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/png;base64,{base64_image}"
                }
            }
        ]
    }
]

results = {}

###############################
# 1. OPENAI — GPT-4o or GPT-5.1
###############################
results["openai"] = litellm.completion(
    model="gpt-5.1",  
    messages=vision_message,
)

###############################
# 2. ANTHROPIC — Claude 3.7 Sonnet
###############################
results["anthropic"] = litellm.completion(
    model="claude-sonnet-4-5-20250929",  
    messages=vision_message,
)

###############################
# 3. GOOGLE — Gemini 1.5 Flash / Pro
###############################
results["google"] = litellm.completion(
    model="gemini/gemini-2.5-flash",    
    messages=vision_message,
)

###############################
# Print all
###############################
for provider, resp in results.items():
    print("=======", provider.upper(), "=======")
    print(resp.choices[0].message.content)
    print()


### 5.3 Visual question answering

VLMs can answer specific questions about images:


In [ ]:
# Using Google Gemini for vision tasks
image_path = "../assets/scones.jpg"
base64_image = encode_image(image_path)

questions = [
    "What food items are in this image?",
    "What colors are prominent in this image?",
    "Describe the setting or environment."
]

for question in questions:
    response = litellm.completion(
        model="gpt-5.1",
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": question
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"}
                    }
                ]
            }
        ]
    )
    print(f"Q: {question}")
    print(f"A: {response.choices[0].message.content}\n")


### 5.5 Processing multiple images

VLMs can process multiple images in a single request:


In [ ]:
# Multiple images in one request
image1 = "../assets/scones.jpg"
image2 = "../assets/muffin.jpg"

response = litellm.completion(
    model="gpt-5.1",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "Compare these two images. What are the main differences?"
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"}
                },
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"}
                }
            ]
        }
    ]
)

print(response.choices[0].message.content)
